In [ ]:
import pandas as pd
import pytest, ipytest
ipytest.autoconfig()    

Question 1: BASH

Question 1.1: How to properly track errors in:

Bash:
    - Use ECHO to print errors
    - Log outputs and errors


Python: 
    - Use exception handelling to catch any errors.
    - Import and use logging function to strucure logging. 


PostgreSQL: 
    - Check log files located in postgresql log folder



Question 1.2

For Slurm job scheduler: 
    - Allocate enough resources (memory, cpu)
    - Make sure there is a time limit
    - Define right input and output path 
    - Specify error and output log file path for debugging
    


Question 1.3

Comments in bash means anything after # is not executed unless '!#' at the start of the script which specifies which interperter should be used to run script; bash, python... 
Comments are important for readability of the code, so users can understand the code logic.
It can be use for whole line, multiple lines or after code.  


Question 2: Data processing

In [ ]:
# File path
file_path = "simple_somatic_mutation.open.BLCA-CN.tsv.gz"

In [ ]:
# Data Exploration
df = pd.read_csv(file_path, sep='\t', compression='gzip') # Read compressed tsv file as pandas dataframe 
print(df.shape)
print(list(df.columns))
df[['icgc_mutation_id', 'icgc_sample_id', 'mutated_from_allele','mutated_to_allele', 'transcript_affected']]


Question 2.1 

In [ ]:
def unique_mutation_counts(df: pd.DataFrame) -> pd.DataFrame: # Defining count function with type hints   
    df_count = df.groupby(['mutated_from_allele','mutated_to_allele']) \
    .agg( counts_of_unique_icgc_mutation_id = ('icgc_mutation_id', 'nunique')) # Grouping mutation patterns and counting number of unique counts 
    return df_count # Return dataframe (series)      

In [ ]:
unique_mutation_counts(df)

Question 2.2 Please find out which icgc_sample_id has the highest and lowest unique icgc_mutation_id
count.

In [ ]:
def sample_unique_mutation_count(df: pd.DataFrame) -> dict:
    
    group = df.groupby('icgc_sample_id')['icgc_mutation_id'].nunique()

    out_dict = { "highest_sample": group.idxmax(), "highest_count": int(group.max()),
        "lowest_sample": group.idxmin(), "lowest_count": int(group.min())}
    
    return out_dict

In [ ]:
sample_unique_mutation_count(df)

Question 2.3: Can you please create some tests in Python to check the functions previously
used in your code.

In [ ]:
def test_unique_mutation_counts():
    # Test dataframe
    df = pd.DataFrame({
        "icgc_mutation_id":    ['MU123', 'MU123', 'MU234', 'MU234', 'MU111'],
        "mutated_from_allele": ["A", "A", "G", "G", "A"],
        "mutated_to_allele":   ["T", "T", "C", "C", "T"]    
    })

    # Run function and save output
    func_out = unique_mutation_counts(df)

    # Expected result   
    expected = pd.DataFrame({"counts_of_unique_icgc_mutation_id": [2, 1]}, 
        index=pd.MultiIndex.from_tuples([("A", "T"), ("G", "C")],
        names=["mutated_from_allele", "mutated_to_allele"] )
        )

    # Check if function output and expected is the same
    pd.testing.assert_frame_equal(func_out, expected)


def test_sample_unique_mutation_count():

    # Test dataframe
    df = pd.DataFrame({
        "icgc_mutation_id":    ['MU123', 'MU123', 'MU234', 'MU234', 'MU111', 'MU456', "MU456"],
        "icgc_sample_id": ["SA123", "SA123", "SA123", "SA123", "SA123", "SA456", "SA456"]
    })

    # Run function and save output
    func_out = sample_unique_mutation_count(df)

    # Check if function output is as expected
    assert func_out["highest_sample"] == "SA123"    
    assert func_out["lowest_sample"] == "SA456"    




In [ ]:
ipytest.run('-vv')

Question 3: Database

Question 3.1:

In [ ]:
select count(*) from gene as g
where g.ID_BIOTYPE == 23;

Question 3.2:

In [ ]:
select g.ENSEMBL_GENE_ID from gene g
where g.GENE_SYMBOL == 'TTTY2';

Question 3.3:

In [ ]:
SELECT CHROMOSOME, count(ID_GENE) as 'number of genes'
from gene
group by CHROMOSOME
order by CHROMOSOME;

Question 3.4:

In [ ]:
select g.GENE_SYMBOL, count(t.ID_TRANSCRIPT) as 'number of transcripts'
from gene g 
join transcript t on g.ID_GENE = t.ID_GENE
where g.GENE_SYMBOL = 'RAI14'
group by  g.GENE_SYMBOL;

or 

-- Just count

select count(t.ID_TRANSCRIPT) as 'number of transcripts'
from gene g 
join transcript t on g.ID_GENE = t.ID_GENE
where g.GENE_SYMBOL = 'RAI14';

Question 3.5:

In [ ]:
select t.ACCESSION from transcript t 
join gene g on t.ID_GENE = g.ID_GENE
where g.ENSEMBL_GENE_ID = 'ENSG00000266960'
and t.IS_CANONICAL = 'y';

Question 3.6:

In [ ]:
select t.ACCESSION from transcript t 
join gene g on t.ID_GENE = g.ID_GENE
where g.GENE_SYMBOL = 'AK1'
and t.ID_BIOTYPE = 23
and t.FLAGS = 'gencode_basic';

Question 3.7:

Use left join from gene table to the subset gene table so all genes are displayed and matching subset genes added.

select * from gene g
left join some_gene s on g.ID_GENE = s.ID_GENE;

Question 3.8:

- Select only specific columns which you need
- Make sure commonly queried fields are indexed 
- Add composite index
- Create materalised views for common aggregations
- Use filters like flags and id's  when querying 

Question 3.9:

You can avoid duplication in the table by adding primary key and unique constraints on specific columns. 

Question 3.10:

Create a foregin key in the transcript table using the gene table primary key index (ID_GENE), the primary key should have a unique index making sure ID_gene in transcript table exists in gene table.